# Build 5k subsequence memmap from 71k memmap

Read the existing **flat-layout** 71k memmap (`subseqs_original_mmap_full`), tile each segment into non-overlapping 5k windows, and write a new memmap in the same flat layout for training with 50/50 batch balance.

**Source:** `MMAP_71K_SOURCE` (X/, target/, weight/, labels.npy, train_inds.npy, val_inds.npy, test_inds.npy).

**Output:** `OUTPUT_DIR_5K` — flat layout (X/, target/, weight/, labels.npy, train_inds.npy, val_inds.npy, test_inds.npy). Positive chunk = at least 1 positive timepoint in that 5k window.

**Requires:** `pip install mmap_ninja`

In [ ]:
import json
import shutil
from pathlib import Path

import numpy as np
from tqdm import tqdm

try:
    from mmap_ninja import RaggedMmap
    HAS_MMAP_NINJA = True
except ImportError:
    HAS_MMAP_NINJA = False
    print("Install mmap_ninja: pip install mmap_ninja")

# ── Config ─────────────────────────────────────────────────────────
MMAP_71K_SOURCE = Path("/home/idies/workspace/Temporary/dpark1/scratch/soen_fusion_zero/subseqs_original_mmap_full")
OUTPUT_DIR_5K = Path("/home/idies/workspace/Temporary/dpark1/scratch/soen_fusion_zero/subseqs_mmap_5k")  # 5k data path

T_SUB_5K = 5000
STRIDE_5K = 5000
MMAP_5K_BATCH = 512
DATA_STEP = 10

print(f"Source (71k): {MMAP_71K_SOURCE}")
print(f"Output (5k):  {OUTPUT_DIR_5K}")
print(f"T_sub_5k={T_SUB_5K}, stride={STRIDE_5K}")

In [ ]:
if not HAS_MMAP_NINJA:
    raise RuntimeError("mmap_ninja required")
if not (MMAP_71K_SOURCE / "X").exists():
    raise FileNotFoundError(f"71k memmap not found: {MMAP_71K_SOURCE / 'X'}")

x_71 = RaggedMmap(str(MMAP_71K_SOURCE / "X"))
t_71 = RaggedMmap(str(MMAP_71K_SOURCE / "target"))
w_71 = RaggedMmap(str(MMAP_71K_SOURCE / "weight"))
labels_71 = np.load(MMAP_71K_SOURCE / "labels.npy")
train_inds_71 = np.load(MMAP_71K_SOURCE / "train_inds.npy")
val_inds_71 = np.load(MMAP_71K_SOURCE / "val_inds.npy") if (MMAP_71K_SOURCE / "val_inds.npy").exists() else np.array([], dtype=np.int64)
test_inds_71 = np.load(MMAP_71K_SOURCE / "test_inds.npy")

n_seq = len(x_71)
T_long = np.asarray(x_71[0]).shape[-1]
n_windows_per_seg = max(0, (T_long - T_SUB_5K) // STRIDE_5K + 1)

print(f"71k: {n_seq} segments, T_long={T_long}")
print(f"5k windows per segment: {n_windows_per_seg}")
print(f"Train indices: {len(train_inds_71)}, val: {len(val_inds_71)}, test: {len(test_inds_71)}")

In [ ]:
if OUTPUT_DIR_5K.exists():
    shutil.rmtree(OUTPUT_DIR_5K)
OUTPUT_DIR_5K.mkdir(parents=True)

X_batch, target_batch, weight_batch = [], [], []
labels_5k = []
train_inds_5k, val_inds_5k, test_inds_5k = [], [], []
mmap_5k_started = False
x_5k = t_5k = w_5k = None

def flush_5k():
    global mmap_5k_started, x_5k, t_5k, w_5k
    if not X_batch:
        return
    if not mmap_5k_started:
        RaggedMmap.from_lists(str(OUTPUT_DIR_5K / "X"), X_batch)
        RaggedMmap.from_lists(str(OUTPUT_DIR_5K / "target"), target_batch)
        RaggedMmap.from_lists(str(OUTPUT_DIR_5K / "weight"), weight_batch)
        mmap_5k_started = True
    else:
        x_5k = x_5k or RaggedMmap(str(OUTPUT_DIR_5K / "X"))
        t_5k = t_5k or RaggedMmap(str(OUTPUT_DIR_5K / "target"))
        w_5k = w_5k or RaggedMmap(str(OUTPUT_DIR_5K / "weight"))
        x_5k.extend(X_batch)
        t_5k.extend(target_batch)
        w_5k.extend(weight_batch)
    X_batch.clear()
    target_batch.clear()
    weight_batch.clear()

global_idx = 0
for split_name, inds_71 in [("train", train_inds_71), ("val", val_inds_71), ("test", test_inds_71)]:
    if len(inds_71) == 0:
        continue
    for idx in tqdm(inds_71, desc=f"5k from {split_name}"):
        X_seg = np.asarray(x_71[idx], dtype=np.float32)
        t_seg = np.asarray(t_71[idx], dtype=np.float32)
        w_seg = np.asarray(w_71[idx], dtype=np.float32)
        for t_start in range(0, T_long - T_SUB_5K + 1, STRIDE_5K):
            t_end = t_start + T_SUB_5K
            X_chunk = np.ascontiguousarray(X_seg[..., t_start:t_end])
            target_chunk = t_seg[t_start:t_end].copy()
            weight_chunk = w_seg[t_start:t_end].copy().astype(np.float32)
            has_pos = np.any(target_chunk > 0.5)
            X_batch.append(X_chunk)
            target_batch.append(target_chunk)
            weight_batch.append(weight_chunk)
            labels_5k.append(1 if has_pos else 0)
            if split_name == "train":
                train_inds_5k.append(global_idx)
            elif split_name == "val":
                val_inds_5k.append(global_idx)
            else:
                test_inds_5k.append(global_idx)
            global_idx += 1
            if len(X_batch) >= MMAP_5K_BATCH:
                flush_5k()
flush_5k()

np.save(OUTPUT_DIR_5K / "labels.npy", np.array(labels_5k, dtype=np.int64))
np.save(OUTPUT_DIR_5K / "train_inds.npy", np.array(train_inds_5k, dtype=np.int64))
np.save(OUTPUT_DIR_5K / "val_inds.npy", np.array(val_inds_5k, dtype=np.int64))
np.save(OUTPUT_DIR_5K / "test_inds.npy", np.array(test_inds_5k, dtype=np.int64))

train_labels = np.array(labels_5k)[train_inds_5k]
total_pos_5k = int(np.sum(train_labels))
total_neg_5k = len(train_labels) - total_pos_5k
pos_weight_5k = (0.5 * (total_pos_5k + total_neg_5k) / total_pos_5k) if total_pos_5k > 0 else 1.0
neg_weight_5k = (0.5 * (total_pos_5k + total_neg_5k) / total_neg_5k) if total_neg_5k > 0 else 1.0

meta_5k = {
    "T_sub": T_SUB_5K,
    "stride_5k": STRIDE_5K,
    "source": str(MMAP_71K_SOURCE),
    "n_total": len(labels_5k),
    "n_train": len(train_inds_5k),
    "n_val": len(val_inds_5k),
    "n_test": len(test_inds_5k),
    "pos_weight": pos_weight_5k,
    "neg_weight": neg_weight_5k,
    "data_step": DATA_STEP,
}
with open(OUTPUT_DIR_5K / "meta.json", "w") as f:
    json.dump(meta_5k, f, indent=2)

print(f"5k memmap: {len(labels_5k):,} chunks -> {OUTPUT_DIR_5K}")
print(f"  train {len(train_inds_5k):,}  val {len(val_inds_5k):,}  test {len(test_inds_5k):,}")
print(json.dumps(meta_5k, indent=2))